In [1]:
import torch
import torch.nn.functional as F

# Import your project modules
import config as cfg
import cnn_utils.data as data_utils
from cnn_utils.model import load_model
from cnn_utils.evaluate import EvalClassification
from cnn_utils.patching import get_dataloader as get_patch_dataloader

# 1. Load the Phase 6 Linear Baseline Model
# (Change to a Phase 7 checkpoint if your Gated Attention numbers were from Phase 7)
MODEL_PATH = cfg.ph6_linear_best 
print(f"Loading model from: {MODEL_PATH}")
model_linear = load_model(MODEL_PATH)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_linear.to(device)
model_linear.eval()

# 2. Load the Dataloaders
print("Loading Lab Dataloaders...")
_, _, test_dl_lab = data_utils.get_3_dataloaders(cfg)

print("Loading OOD Dataloaders...")
_, test_dl_ood = data_utils.get_ood_dataloaders(cfg)

# 3. Initialize Evaluators
eval_lab = EvalClassification(cfg, model_linear, test_dl_lab)
eval_ood = EvalClassification(cfg, model_linear, test_dl_ood)

/home/takayuki/.local/lib/python3.10/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(
/home/takayuki/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
/home/takayuki/.local/lib/python3.10/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Outp

Loading model from: /home/takayuki/Desktop/summer2025/plants/training/phase6/models/best_model_07-14_14-56--10_p6_linear_convnextv2_tiny.fcmae_ft_in22k_in1k.pth
Model loaded successfully from /home/takayuki/Desktop/summer2025/plants/training/phase6/models/best_model_07-14_14-56--10_p6_linear_convnextv2_tiny.fcmae_ft_in22k_in1k.pth
Loading Lab Dataloaders...
 Total dataset size 	: 230
 Train dataset size 	: 138
 Val dataset size 	: 46
 Test dataset size 	: 46

Loading OOD Dataloaders...
Found 10 classes locally: ['Hydrocharis morsus-ranae', 'Myriophyllum spicatum', 'Nitellopsis obtusa', 'Nuphar variegata', 'Potamogeton crispus', 'Potamogeton gramineus', 'Potamogeton illinoensis', 'Potamogeton richardsonii', 'Potamogeton robbinsii', 'Ranunculus aquatilis']

OOD split complete:
  OOD Validation set size: 16
  OOD Test set size    : 17



In [2]:
def run_baseline_evaluation():
    print("--- EVALUATING LINEAR CLASSIFIER (NO YNLT) ---")
    
    # Lab Data
    print("\nRunning on Lab Test Set...")
    eval_lab.evaluate() # Uses default forward pass
    lab_acc = eval_lab.get_accuracy(verbose=False) * 100
    lab_metrics = eval_lab.get_binary_metrics(display=False)
    lab_fnr = lab_metrics['FNR'] * 100
    
    # OOD Data
    print("Running on OOD Test Set...")
    eval_ood.evaluate()
    ood_acc = eval_ood.get_accuracy(verbose=False) * 100
    ood_metrics = eval_ood.get_binary_metrics(display=False)
    ood_fnr = ood_metrics['FNR'] * 100
    
    print("\n=== BASELINE RESULTS FOR TABLE ===")
    print(f"Lab Accuracy : {lab_acc:.1f}%")
    print(f"Lab FNR      : {lab_fnr:.1f}%")
    print(f"OOD Accuracy : {ood_acc:.1f}%")
    print(f"OOD FNR      : {ood_fnr:.1f}%")

run_baseline_evaluation()

--- EVALUATING LINEAR CLASSIFIER (NO YNLT) ---

Running on Lab Test Set...


  -- Evaluating: 100%|██████████| 46/46 [00:05<00:00,  8.51it/s]


Running on OOD Test Set...


  -- Evaluating: 100%|██████████| 17/17 [00:01<00:00,  9.81it/s]


=== BASELINE RESULTS FOR TABLE ===
Lab Accuracy : 84.8%
Lab FNR      : 30.0%
OOD Accuracy : 47.1%
OOD FNR      : 62.5%


In [3]:
def calculate_confusion(logits):
    """Calculates confusion level (0 to 3) based on config thresholds."""
    probs = F.softmax(logits, dim=1)[0]
    top_probs, top_indices = torch.topk(probs, 2)
    
    confidence = top_probs[0].item()
    margin = (top_probs[0] - top_probs[1]).item()
    entropy = -torch.sum(probs * torch.log2(probs + 1e-9)).item()
    
    confusion_level = 0
    if confidence < cfg.confidence_threshold: confusion_level += cfg.confidence_weight
    if margin < cfg.margin_threshold: confusion_level += cfg.margin_weight
    if entropy > cfg.entropy_threshold: confusion_level += cfg.entropy_weight
        
    return confusion_level, top_indices[0].item()

def ynlt_predict(images, label_item):
    """Custom predict function for the YNLT ensemble."""
    with torch.no_grad():
        # 1. Initial Look
        base_logits = model_linear(images)
        confusion_level, candidate_pred = calculate_confusion(base_logits)
        
        # 2. Check if a second look is needed
        if confusion_level < cfg.critical_confusion_level:
            return {'predicted_class_idx': candidate_pred}
        
        # 3. YNLT Second Look (Patching & Ensemble)
        # Convert tensor back to image format for your patching utility
        img_np = data_utils.to_uint8_img(cfg, images) 
        patch_dl = get_patch_dataloader(img_np, label=label_item, batch_size=1)
        
        class_votes = torch.zeros(cfg.CANONICAL_NUM_CLASSES).to(device)
        
        for patch_imgs, _ in patch_dl:
            patch_imgs = patch_imgs.to(device)
            patch_logits = model_linear(patch_imgs)
            
            p_conf_level, p_pred = calculate_confusion(patch_logits)
            
            # Weight formula from the paper: W_i = 4 - c_i
            weight = 4 - p_conf_level 
            
            # Aggregate probabilities weighted by patch confidence
            patch_probs = F.softmax(patch_logits, dim=1)[0]
            class_votes += (patch_probs * weight)
            
        final_pred = torch.argmax(class_votes).item()
        return {'predicted_class_idx': final_pred}

def run_ynlt_evaluation():
    print("--- EVALUATING LINEAR CLASSIFIER + YNLT ---")
    
    # Lab Data
    print("\nRunning YNLT on Lab Test Set...")
    eval_lab.evaluate(predict_function=ynlt_predict)
    lab_acc = eval_lab.get_accuracy(verbose=False) * 100
    lab_metrics = eval_lab.get_binary_metrics(display=False)
    lab_fnr = lab_metrics['FNR'] * 100
    
    # OOD Data
    print("Running YNLT on OOD Test Set...")
    eval_ood.evaluate(predict_function=ynlt_predict)
    ood_acc = eval_ood.get_accuracy(verbose=False) * 100
    ood_metrics = eval_ood.get_binary_metrics(display=False)
    ood_fnr = ood_metrics['FNR'] * 100
    
    print("\n=== YNLT RESULTS FOR TABLE ===")
    print(f"Lab Accuracy : {lab_acc:.1f}%")
    print(f"Lab FNR      : {lab_fnr:.1f}%")
    print(f"OOD Accuracy : {ood_acc:.1f}%")
    print(f"OOD FNR      : {ood_fnr:.1f}%")

run_ynlt_evaluation()

--- EVALUATING LINEAR CLASSIFIER + YNLT ---

Running YNLT on Lab Test Set...


  -- Evaluating: 100%|██████████| 46/46 [00:31<00:00,  1.48it/s]


Running YNLT on OOD Test Set...


  -- Evaluating: 100%|██████████| 17/17 [00:28<00:00,  1.70s/it]


=== YNLT RESULTS FOR TABLE ===
Lab Accuracy : 89.1%
Lab FNR      : 30.0%
OOD Accuracy : 58.8%
OOD FNR      : 62.5%


In [ ]:
# 1. Load the Gated Attention Model
# Using your config's best_model_path (ph7_gated_attn_best_ood_acc)
GATED_MODEL_PATH = cfg.best_model_path 
print(f"Loading Gated Attention model from: {GATED_MODEL_PATH}")

model_gated = load_model(GATED_MODEL_PATH)
model_gated.to(device)
model_gated.eval()

# 2. Initialize New Evaluators
eval_lab_gated = EvalClassification(cfg, model_gated, test_dl_lab)
eval_ood_gated = EvalClassification(cfg, model_gated, test_dl_ood)

# 3. Evaluate Baseline (No YNLT)
print("\n--- EVALUATING GATED ATTENTION CLASSIFIER (NO YNLT) ---")

print("\nRunning on Lab Test Set...")
eval_lab_gated.evaluate() 
lab_acc_g = eval_lab_gated.get_accuracy(verbose=False) * 100
lab_fnr_g = eval_lab_gated.get_binary_metrics(display=False)['FNR'] * 100

print("Running on OOD Test Set...")
eval_ood_gated.evaluate()
ood_acc_g = eval_ood_gated.get_accuracy(verbose=False) * 100
ood_fnr_g = eval_ood_gated.get_binary_metrics(display=False)['FNR'] * 100

print("\n=== GATED ATTENTION (BASE) RESULTS ===")
print(f"Table Expected -> Lab Acc: 84%, Lab FNR: 10%, OOD Acc: 57%, OOD FNR: 13%")
print(f"Actual Output  -> Lab Acc: {lab_acc_g:.1f}%, Lab FNR: {lab_fnr_g:.1f}%, OOD Acc: {ood_acc_g:.1f}%, OOD FNR: {ood_fnr_g:.1f}%")

Loading Gated Attention model from: /home/takayuki/Desktop/summer2025/plants/training/phase7/models/best_ood_acc_model_07-31_04-19--55_ph7_gated_attn_actual_convnextv2_tiny.fcmae_ft_in22k_in1k.pth
Model loaded successfully from /home/takayuki/Desktop/summer2025/plants/training/phase7/models/best_ood_acc_model_07-31_04-19--55_ph7_gated_attn_actual_convnextv2_tiny.fcmae_ft_in22k_in1k.pth

--- EVALUATING GATED ATTENTION CLASSIFIER (NO YNLT) ---

Running on Lab Test Set...


  -- Evaluating: 100%|██████████| 46/46 [00:04<00:00, 10.32it/s]


Running on OOD Test Set...


  -- Evaluating: 100%|██████████| 17/17 [00:01<00:00, 10.25it/s]


=== GATED ATTENTION (BASE) RESULTS ===
Table Expected -> Lab Acc: 84%, Lab FNR: 10%, OOD Acc: 57%, OOD FNR: 13%
Actual Output  -> Lab Acc: 87.0%, Lab FNR: 10.0%, OOD Acc: 52.9%, OOD FNR: 12.5%


In [ ]:
def ynlt_predict_gated(images, label_item):
    """Custom predict function for YNLT using the Gated Attention model."""
    with torch.no_grad():
        # 1. Initial Look
        base_logits = model_gated(images)
        confusion_level, candidate_pred = calculate_confusion(base_logits)
        
        # 2. Check if a second look is needed
        if confusion_level < cfg.critical_confusion_level:
            return {'predicted_class_idx': candidate_pred}
        
        # 3. YNLT Second Look
        img_np = data_utils.to_uint8_img(cfg, images) 
        patch_dl = get_patch_dataloader(img_np, label=label_item, batch_size=1)
        
        class_votes = torch.zeros(cfg.CANONICAL_NUM_CLASSES).to(device)
        
        for patch_imgs, _ in patch_dl:
            patch_imgs = patch_imgs.to(device)
            patch_logits = model_gated(patch_imgs)
            
            p_conf_level, p_pred = calculate_confusion(patch_logits)
            weight = 4 - p_conf_level 
            
            patch_probs = F.softmax(patch_logits, dim=1)[0]
            class_votes += (patch_probs * weight)
            
        final_pred = torch.argmax(class_votes).item()
        return {'predicted_class_idx': final_pred}

print("--- EVALUATING GATED ATTENTION + YNLT ---")

# Lab Data
print("\nRunning YNLT on Lab Test Set...")
eval_lab_gated.evaluate(predict_function=ynlt_predict_gated)
lab_acc_gy = eval_lab_gated.get_accuracy(verbose=False) * 100
lab_fnr_gy = eval_lab_gated.get_binary_metrics(display=False)['FNR'] * 100

# OOD Data
print("Running YNLT on OOD Test Set...")
eval_ood_gated.evaluate(predict_function=ynlt_predict_gated)
ood_acc_gy = eval_ood_gated.get_accuracy(verbose=False) * 100
ood_fnr_gy = eval_ood_gated.get_binary_metrics(display=False)['FNR'] * 100

print("\n=== GATED ATTENTION + YNLT RESULTS ===")
print(f"Table Expected -> Lab Acc: 89%, Lab FNR: 10%, OOD Acc: 63%, OOD FNR: 13%")
print(f"Actual Output  -> Lab Acc: {lab_acc_gy:.1f}%, Lab FNR: {lab_fnr_gy:.1f}%, OOD Acc: {ood_acc_gy:.1f}%, OOD FNR: {ood_fnr_gy:.1f}%")

--- EVALUATING GATED ATTENTION + YNLT ---

Running YNLT on Lab Test Set...


  -- Evaluating: 100%|██████████| 46/46 [00:27<00:00,  1.67it/s]


Running YNLT on OOD Test Set...


  -- Evaluating: 100%|██████████| 17/17 [00:28<00:00,  1.66s/it]


=== GATED ATTENTION + YNLT RESULTS ===
Table Expected -> Lab Acc: 89%, Lab FNR: 10%, OOD Acc: 63%, OOD FNR: 13%
Actual Output  -> Lab Acc: 91.3%, Lab FNR: 10.0%, OOD Acc: 29.4%, OOD FNR: 12.5%


In [7]:
# Create a dictionary of the most likely checkpoint candidates
checkpoint_candidates = {
    "Phase 5 Best": cfg.p54b_gated_attn,
    "Phase 6 Best": cfg.ph6_gated_attn_best,
    "Phase 6.2 Best Acc": cfg.ph62_gattn_best_acc,
    "Phase 7 Best OOD Loss": cfg.ph7_gated_attn_best_ood_loss,
    "Phase 7 Final": cfg.ph7_gated_attn
}

print("=== HUNTING FOR THE 84% / 57% CHECKPOINT ===\n")

for name, path in checkpoint_candidates.items():
    print(f"Testing: {name}")
    try:
        # Load the model
        temp_model = load_model(path, verbose=False)
        temp_model.to(device)
        temp_model.eval()
        
        # Initialize evaluators
        temp_eval_lab = EvalClassification(cfg, temp_model, test_dl_lab)
        temp_eval_ood = EvalClassification(cfg, temp_model, test_dl_ood)
        
        # Evaluate Lab
        temp_eval_lab.evaluate()
        lab_acc = temp_eval_lab.get_accuracy(verbose=False) * 100
        lab_fnr = temp_eval_lab.get_binary_metrics(display=False)['FNR'] * 100
        
        # Evaluate OOD
        temp_eval_ood.evaluate()
        ood_acc = temp_eval_ood.get_accuracy(verbose=False) * 100
        ood_fnr = temp_eval_ood.get_binary_metrics(display=False)['FNR'] * 100
        
        print(f"--> Results: Lab Acc: {lab_acc:.1f}%, Lab FNR: {lab_fnr:.1f}%, OOD Acc: {ood_acc:.1f}%, OOD FNR: {ood_fnr:.1f}%\n")
        
        if round(lab_acc) == 84 and round(ood_acc) == 57:
            print(f"*** MATCH FOUND! {name} is the model from your paper draft! ***\n")
            
    except Exception as e:
        print(f"Failed to load or test {name}: {e}\n")

=== HUNTING FOR THE 84% / 57% CHECKPOINT ===

Testing: Phase 5 Best


  -- Evaluating: 100%|██████████| 17/17 [00:01<00:00, 11.84it/s]


--> Results: Lab Acc: 87.0%, Lab FNR: 10.0%, OOD Acc: 64.7%, OOD FNR: 12.5%

Testing: Phase 6 Best


  -- Evaluating: 100%|██████████| 17/17 [00:01<00:00, 11.85it/s]


--> Results: Lab Acc: 87.0%, Lab FNR: 10.0%, OOD Acc: 76.5%, OOD FNR: 12.5%

Testing: Phase 6.2 Best Acc


...

--> Results: Lab Acc: 93.5%, Lab FNR: 20.0%, OOD Acc: 58.8%, OOD FNR: 25.0%

Testing: Phase 7 Best OOD Loss


  -- Evaluating: 100%|██████████| 17/17 [00:01<00:00, 12.93it/s]


--> Results: Lab Acc: 89.1%, Lab FNR: 10.0%, OOD Acc: 52.9%, OOD FNR: 25.0%

Testing: Phase 7 Final


  -- Evaluating: 100%|██████████| 17/17 [00:01<00:00, 11.93it/s]

--> Results: Lab Acc: 84.8%, Lab FNR: 20.0%, OOD Acc: 64.7%, OOD FNR: 12.5%

